# multitask-learning-framework walkthrough

End-to-end demos for the NLP and CV tracks plus a side-by-side comparison of
the three loss-weighting strategies.

What this notebook covers:
1. Build a tiny shared-trunk model (NLP).
2. Train it under uniform / uncertainty / GradNorm weighting.
3. Inspect per-task loss curves and task weight trajectories.
4. Repeat for the vision demo (cls + seg).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
from src.core.mtl_model import MTLModel
from src.core.tasks import Task, make_ce_loss
from src.core.loss_weighting import UniformWeighter, UncertaintyWeighter, GradNormWeighter
from src.core.trainer import MTLTrainer
from src import visualize

## 1. Toy MTL setup (NLP-flavoured)

We use a small linear trunk and two heads to keep things fast. Replace with
BertBackbone + nlp.heads for the real demo (see `examples/run_nlp.py`).

In [ ]:
class Trunk(nn.Module):
    def __init__(self, dim=16):
        super().__init__()
        self.last_layer = nn.Linear(8, dim)
    def forward(self, x): return self.last_layer(x)

def build(weighter_cls, seed=0):
    torch.manual_seed(seed)
    trunk = Trunk(16)
    tasks = [
        Task('cls_a', nn.Linear(16, 3), make_ce_loss()),
        Task('cls_b', nn.Linear(16, 5), make_ce_loss()),
    ]
    model = MTLModel(trunk, tasks)
    weighter = weighter_cls([t.name for t in tasks])
    optim = torch.optim.SGD(
        list(model.parameters()) + list(weighter.parameters()), lr=0.05
    )
    return MTLTrainer(model, weighter, optim)

def make_loader(n_steps=200, bs=8):
    for _ in range(n_steps):
        yield {
            'inputs': torch.randn(bs, 8),
            'targets': {
                'cls_a': torch.randint(0, 3, (bs,)),
                'cls_b': torch.randint(0, 5, (bs,)),
            },
        }

## 2. Train under each weighter

In [ ]:
histories = {}
for name, cls in [('uniform', UniformWeighter), ('uncertainty', UncertaintyWeighter), ('gradnorm', GradNormWeighter)]:
    trainer = build(cls)
    histories[name] = trainer.fit(list(make_loader(200, 8)), epochs=1, log_every=50)

## 3. Visualise

In [ ]:
import os
os.makedirs('figs', exist_ok=True)
for name, h in histories.items():
    visualize.plot_per_task_losses(h, ['cls_a', 'cls_b'], f'figs/{name}_losses.png')
    visualize.plot_task_weights(h, ['cls_a', 'cls_b'], f'figs/{name}_weights.png')
print('saved plots in figs/')

## 4. Vision demo (sketch)

See `examples/run_vision.py` for the real run with ResNet50 + classification +
segmentation. The full run takes a few minutes on CPU.